In [ ]:
# =====================================================
# STEP 0: Mount Google Drive
# =====================================================
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir("/content/drive/MyDrive/")

# =====================================================
# STEP 1: Extract frames from video
# =====================================================
import cv2
import numpy as np

VIDEO_FILE = 'video.mp4'
FPS_OBJETIVO = 24
SECONDS = 10
TOTAL_FRAMES = FPS_OBJETIVO * SECONDS   # = 240
WIDTH = 320
HEIGHT = 240

vidcap = cv2.VideoCapture(VIDEO_FILE)
fps_original = vidcap.get(cv2.CAP_PROP_FPS)
total_frames_video = int(vidcap.get(cv2.CAP_PROP_FRAME_COUNT))

print(f"Original video FPS: {fps_original}")
print(f"Total frames in video: {total_frames_video}")
print(f"Duration: {total_frames_video/fps_original:.2f} seconds")

indexes_to_take = np.linspace(0, min(total_frames_video - 1,
                               int(fps_original * SECONDS) - 1),
                               TOTAL_FRAMES, dtype=int)

frames_rgb565 = []

for i, frame_idx in enumerate(indexes_to_take):
    vidcap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    success, frame = vidcap.read()
    if not success:
        print(f"Error reading frame {frame_idx}, using black frame")
        frame = np.zeros((HEIGHT, WIDTH, 3), dtype=np.uint8)

    frame_resized = cv2.resize(frame, (WIDTH, HEIGHT))
    frame_rgb = cv2.cvtColor(frame_resized, cv2.COLOR_BGR2RGB)

    r = (frame_rgb[:, :, 0].astype(np.uint16) >> 3) & 0x1F
    g = (frame_rgb[:, :, 1].astype(np.uint16) >> 2) & 0x3F
    b = (frame_rgb[:, :, 2].astype(np.uint16) >> 3) & 0x1F
    rgb565 = (r << 11) | (g << 5) | b

    frames_rgb565.append(rgb565)

    if i % 24 == 0:
        print(f"  Processed frame {i+1}/{TOTAL_FRAMES}")

vidcap.release()
print(f"\nTotal frames processed: {len(frames_rgb565)}")
print(f"Estimated total memory: {len(frames_rgb565) * WIDTH * HEIGHT * 2 / 1024 / 1024:.1f} MB")

# =====================================================
# STEP 2: Generate the Assembly file
# =====================================================
print("\nGenerating video_data.s file ...")

OUTPUT_FILE = "video_data.s"

with open(OUTPUT_FILE, "w") as f:
    f.write(".data\n.global video_data_start\n.global video_data_end\nvideo_data_start:\n")

    for i in range(TOTAL_FRAMES):
        frame = frames_rgb565[i]
        for row in range(HEIGHT):
            for col in range(0, WIDTH, 2):
                p0 = int(frame[row, col])
                p1 = int(frame[row, col + 1])
                word_val = (p0 << 16) | p1
                f.write(f".word 0x{word_val:08X}\n")

    f.write("video_data_end:\n.word 0x00000000\n")

print(f"File generated: {OUTPUT_FILE}")
print(f"Approximate .s size: {os.path.getsize(OUTPUT_FILE)/1024/1024:.1f} MB")
print("\nDone! Copy video_data.s to your project in VS Code.")